# Main table — all-method embedding distillation

Notebook này chỉ chạy thực nghiệm của **bảng kết quả chính**: tuần tự cả 8 method trong repo với **một cặp teacher → student
chọn ở cell 1** (`PAIR`). Ba cặp được cấu hình sẵn:

| `PAIR` | Teacher | Student | Câu hỏi |
| --- | --- | --- | --- |
| `qwen3_0.6b_to_minilm_h384` | `Qwen/Qwen3-Embedding-0.6B` (1024-d, last-token) | `MiniLMv2-L6-H384` (384-d, 6 layer) | student nhỏ, ít flow step |
| `bge_m3_to_minilm_h768` | `BAAI/bge-m3` (XLM-R, 1024-d, CLS) | `MiniLMv2-L6-H768` (768-d, 6 layer) | khác teacher family |
| `qwen3_4b_to_bert_base` | `Qwen/Qwen3-Embedding-4B` (2560-d, last-token) | `bert-base-uncased` (768-d, 12 layer) | capacity gap lớn |

- **Methods:** SimCSE-only, TALAS, GeoODE-KD, RKD, CDM, DSKD, EMO, Stella, **PCA + MSE**

**PCA + MSE** là recipe distillation của sentence-transformers ≤ v5.4 (PCA teacher về `d_S`, MSE, không
InfoNCE, không gauge). Nó chạy qua đúng code path của `geoode` với ba flag
(`--endpoint_loss mse --lambda_ctr 0 --no-gauge_align`), nên cùng cache teacher, cùng PCA, cùng
lịch train — thứ duy nhất khác dòng `geoode` là loss và hai thành phần bị tắt.

Corpus train chọn ở cùng cell 1 (`DATASET`), độc lập với `PAIR`:

| `DATASET` | Nội dung | Dùng để |
| --- | --- | --- |
| `talas_15k` | ~15k câu EMOTION/WiC/STS-B | tái lập setup của bài TALAS |
| `100k` | 102,361 dòng benchmark train | mốc trước khi thêm MS MARCO |
| `150k` | 100k base + 25k query + 25k passage MS MARCO | mặc định |
| `200k` | 100k base + 50k query + 50k passage MS MARCO | rung trên của data-scaling |

`150k` và `200k` không nằm trong git; cell 3 tự dựng chúng bằng
`scripts/build_train_corpus.py` khi thiếu (`AUTO_FETCH_DATA`). Base 100k giống hệt
nhau ở cả hai mức và khối MS MARCO của `150k` là prefix của `200k`, nên so sánh
giữa hai mức chỉ khác kích thước corpus. `RUN_NAME` mang cả `PAIR` lẫn `DATASET`
nên hai run khác corpus không ghi đè nhau.

Mỗi cặp mang theo pooling của teacher (`--teacher_pooling`: Qwen3 đọc token cuối,
BGE-M3 đọc CLS) và marker sub-word của tokenizer teacher (`--teacher_special_token`:
`Ġ` cho byte-level BPE của Qwen3, `▁` cho SentencePiece của BGE-M3) — cả hai được
truyền cho mọi method nên không phải chỉnh tay. Cache teacher được đặt tên theo cặp
và dataset, đồng thời mang metadata (teacher, pooling, corpus) nên chạy nhầm cache
của teacher hoặc corpus khác sẽ bị từ chối ngay lúc khởi động.

**SimCSE-only** và **RKD** là hai mốc để đọc các method còn lại. SimCSE-only là control không distill: cùng student, cùng corpus, cùng lịch train và cùng pooling, chỉ bỏ phần teacher (command vẫn mang `--teacher_model`/`--teacher_pooling` để config ghi lại nó là control của cặp nào, nhưng distiller **không tải teacher**) và giữ lại đúng InfoNCE mà các objective kia vốn đã chứa — nên phần điểm một method vượt lên trên dòng này chính là thứ tín hiệu teacher mang lại. **RKD** (Park et al., 2019) là mốc quan hệ: teacher chỉ giám sát khoảng cách và góc giữa các mẫu ở layer cuối, không nói gì về đường đi qua từng layer.

Mỗi method chạy trong một Python process riêng, có checkpoint, metrics và log riêng. Notebook dùng batch size thận trọng cho các method phải giữ teacher trên GPU; có thể tăng ở cell cấu hình nếu GPU còn nhiều VRAM. Nên dùng GPU hỗ trợ BF16; cặp Qwen3-4B cần ít nhất 24 GiB VRAM, hai cặp còn lại nhẹ hơn nhiều. Hai GPU sẽ được codebase tự động chia teacher/student.

Bảng kết quả có bốn family: classification, pair, STS và **retrieval**
(ArguAna/FiQA/SCIDOCS, nDCG@10). Retrieval chỉ chấm ở lần eval test — ba benchmark
này không có validation qrels và phải encode ~92k document. Đây là family duy nhất
hoàn toàn zero-shot: MS MARCO trong corpus train chỉ là nguồn *train*, ba benchmark
kia không đóng góp dòng nào. Tắt bằng `EVAL_RETRIEVAL = False` ở cell 1.


In [9]:
# 1. Cấu hình thí nghiệm. Chỉnh các giá trị trong cell này trước khi chạy.
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"

# Ba cặp teacher -> student của bài. Mỗi cặp mang theo hai thứ phụ thuộc vào
# *family* của teacher, để không method nào phải chỉnh tay:
#   teacher_pooling:       cách đọc sentence vector của teacher. Qwen3-Embedding là
#                          decoder nên đọc token cuối; BGE-M3 là encoder XLM-R, đọc CLS.
#   teacher_special_token: marker sub-word mà CDM strip trước khi so token string
#                          ("Ġ" byte-level BPE của Qwen3, "▁" SentencePiece của BGE-M3).
#                          EMO đọc giá trị này như BOS token của teacher; BGE-M3 có
#                          "<s>" thật, Qwen3 không có BOS nên EMO giữ default của nó.
PAIRS = {
    "qwen3_0.6b_to_minilm_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "teacher_pooling": "last_token",
        "teacher_special_token": "Ġ",
        "emo_teacher_special_token": None,
        "min_vram_gib": 12,
        "note": "student nhỏ (384-d, 6 layer): GeoODE chỉ có 6 flow step",
    },
    "bge_m3_to_minilm_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "teacher_pooling": "cls",
        "teacher_special_token": "▁",
        "emo_teacher_special_token": "<s>",
        "min_vram_gib": 12,
        "note": "khác teacher family (XLM-R encoder, CLS pooling, SentencePiece)",
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "teacher_pooling": "last_token",
        "teacher_special_token": "Ġ",
        "emo_teacher_special_token": None,
        "min_vram_gib": 24,
        "note": "capacity gap lớn (2560-d teacher 4B params -> 768-d student)",
    },
}
PAIR = "bge_m3_to_minilm_h768"  # <- chọn 1 trong 3 key của PAIRS

PAIR_CONFIG = PAIRS[PAIR]
TEACHER_MODEL = PAIR_CONFIG["teacher"]
STUDENT_MODEL = PAIR_CONFIG["student"]
TEACHER_POOLING = PAIR_CONFIG["teacher_pooling"]
TEACHER_SPECIAL_TOKEN = PAIR_CONFIG["teacher_special_token"]
EMO_TEACHER_SPECIAL_TOKEN = PAIR_CONFIG["emo_teacher_special_token"]
# Các mức của thang data-scaling. Mọi method trong một run phải dùng chung một
# mức, và mức lớn *mở rộng* mức nhỏ (base 100k giống hệt, khối MS MARCO là prefix),
# nên so sánh giữa các mức chỉ khác kích thước corpus chứ không khác nội dung.
#   build: lệnh dựng lại file nếu chưa có (None = file đã nằm sẵn trong repo).
DATASETS = {
    "talas_15k": {
        "path": Path("data/train_set/merged_3_data_5k_each.csv"),
        "build": None,
        "note": "setup của bài TALAS: ~15k câu từ EMOTION/WiC/STS-B",
    },
    "100k": {
        "path": Path("data/train_set/train_100k.csv"),
        "build": None,
        "note": "102,361 dòng benchmark train — KHÔNG phải base 100k của 150k/200k",
    },
    "150k": {
        "path": Path("data/train_set/train_150k.csv"),
        "build": "scripts/build_train_corpus.py --total 150000",
        "note": "100k base + 25k MS MARCO query + 25k MS MARCO passage",
    },
    "200k": {
        "path": Path("data/train_set/train_200k.csv"),
        "build": "scripts/build_train_corpus.py --total 200000",
        "note": "100k base + 50k MS MARCO query + 50k MS MARCO passage",
    },
}
DATASET = "100k"  # <- chọn 1 trong 4 key của DATASETS

DATASET_CONFIG = DATASETS[DATASET]
TRAIN_DATA_REL = DATASET_CONFIG["path"]
MAX_LENGTH = 256
EPOCHS = 5
NUM_WORKERS = 2
CUDA_VISIBLE_DEVICES = "0,1"
STOP_ON_ERROR = True
# Một công tắc cho mọi bước đánh giá và hiệu chỉnh.
#   False: eval từng epoch trên validation, ngưỡng pair sweep trên validation,
#          điểm test cuối vẫn là held-out.
#   True:  eval từng epoch trên test và sweep ngưỡng ngay trên test, bỏ hẳn
#          validation. Bám sát mục tiêu và nhanh hơn, nhưng không còn con số nào
#          là held-out — mọi lựa chọn epoch/ngưỡng đều đã nhìn thấy test.
EVAL_ON_TEST_EACH_EPOCH = True
PAIR_THRESHOLD_SOURCE = "test" if EVAL_ON_TEST_EACH_EPOCH else "validation"
# Chu kỳ eval từng epoch: N = eval sau mỗi N epoch, 0 = tắt hẳn (chỉ eval test cuối).
EVAL_EVERY = 0
# Chấm ArguAna/FiQA/SCIDOCS (nDCG@10) trong lần eval test. Đây là family duy nhất
# hoàn toàn zero-shot: MS MARCO chỉ là nguồn *train*, ba benchmark này không đóng
# góp dòng nào vào corpus. Đổi lại nó phải encode ~92k document nên chậm hơn hẳn
# ba family còn lại cộng lại.
EVAL_RETRIEVAL = False
# Corpus 150k/200k và ba benchmark retrieval đều không nằm trong git (quá lớn).
# Trên Colab repo được clone mới mỗi lần nên notebook tự dựng chúng.
AUTO_FETCH_DATA = True
SAVE_TO_GOOGLE_DRIVE = False

# Preset ưu tiên chạy được trên GPU 24-40 GiB với cặp nặng nhất (Qwen3-4B); hai cặp
# còn lại nhẹ hơn nhiều nên có thể tăng batch của cdm/dskd/emo/stella. TALAS/GeoODE/
# RKD cache teacher rồi giải phóng teacher nên không phụ thuộc kích thước teacher.
# Thứ tự dict cũng là thứ tự chạy: simcse chạy đầu vì nó không tải teacher và không
# cần cache, nên nếu data/eval có vấn đề thì hỏng ngay trong vài phút thay vì sau
# khi đã cache xong teacher; talas/geoode/rkd đứng liền nhau để dùng chung một
# lần chạy teacher. Batch size của rkd và simcse là số negative/quan hệ trong
# batch, không chỉ là nút chỉnh bộ nhớ: cả hai đo mọi thứ theo cặp trong batch.
# Comment một dòng để bỏ method đó khỏi bảng (ví dụ khi chỉ cần chạy bổ sung).
# Key là tên dòng trong bảng (và tên thư mục output). Một dòng có thể là *biến thể*
# của một method: "method" nói --method nào được gọi, "args" là flag thêm vào.
# pca_mse = recipe sentence-transformers <= v5.4 (PCA + MSE, không InfoNCE, không
# gauge) chạy qua code path của geoode, cùng HP với dòng geoode để chỉ khác loss.
METHOD_SETTINGS = {
    # "simcse":  {"batch_size": 64, "learning_rate": 2e-5},
    # "talas":   {"batch_size": 64, "learning_rate": 2e-5},
    # "geoode":  {"batch_size": 64, "learning_rate": 5e-5},
    "pca_mse": {"method": "geoode", "batch_size": 64, "learning_rate": 5e-5,
                "args": ["--endpoint_loss", "mse", "--lambda_ctr", "0", "--no-gauge_align"]},
    # "rkd":     {"batch_size": 64, "learning_rate": 2e-5},
    # "cdm":     {"batch_size": 64, "learning_rate": 2e-5},
    # "dskd":    {"batch_size": 64, "learning_rate": 2e-5},
    # "emo":     {"batch_size": 64, "learning_rate": 1e-5},
    # "stella":  {"batch_size": 64, "learning_rate": 5e-5},
}

RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
RUN_NAME = f"{PAIR}_{DATASET}_all_methods_{RUN_STAMP}"
# Đặt RUN_NAME thành tên một run cũ để chạy tiếp vào đó (ví dụ chạy bổ sung method
# còn thiếu). Cell 5 bỏ qua method đã có final test và dừng hẳn khi gặp run dở
# dang, nên không thể trộn hai cấu hình vào một thư mục.
print(f"Pair: {PAIR} — {PAIR_CONFIG['note']}")
print(f"  teacher: {TEACHER_MODEL} (pooling={TEACHER_POOLING}, marker={TEACHER_SPECIAL_TOKEN!r})")
print(f"  student: {STUDENT_MODEL}")
print(f"Dataset: {DATASET} — {DATASET_CONFIG['note']}")
print(f"Run name: {RUN_NAME}")
print(f"Methods: {', '.join(METHOD_SETTINGS)}")


Pair: bge_m3_to_minilm_h768 — khác teacher family (XLM-R encoder, CLS pooling, SentencePiece)
  teacher: BAAI/bge-m3 (pooling=cls, marker='▁')
  student: nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base
Dataset: 100k — 102,361 dòng benchmark train — KHÔNG phải base 100k của 150k/200k
Run name: bge_m3_to_minilm_h768_100k_all_methods_20260829-224211
Methods: pca_mse


In [ ]:
# 2. Dùng repo hiện tại nếu notebook nằm trong repo; nếu không thì clone từ GitHub.
import subprocess
import sys

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file() and (cwd / "distiller.py").is_file():
    PROJECT_DIR = cwd
else:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if PROJECT_DIR.exists():
        assert (PROJECT_DIR / "main.py").is_file(), (
            f"Thư mục đã tồn tại nhưng không phải repo hợp lệ: {PROJECT_DIR}"
        )
        print(f"Reuse existing clone: {PROJECT_DIR}")
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

assert (PROJECT_DIR / "requirements.txt").is_file()
print(f"Project directory: {PROJECT_DIR}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)
print("Dependencies installed.")


: 

In [ ]:
# 3. Chọn nơi lưu output và kiểm tra GPU/data trước khi tải model lớn.
import os

import torch

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

RUN_ROOT = OUTPUT_BASE / RUN_NAME
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
RETRIEVAL_DIR = PROJECT_DIR / "data" / "test_set" / "retrieval"


def run_script(argv, what):
    """Chạy một script trong repo và dừng hẳn nếu nó fail."""
    print(f"[data] {what}: python3 {' '.join(argv)}")
    subprocess.run([sys.executable, *argv], cwd=PROJECT_DIR, check=True)


for split in ("train_set", "val_set", "test_set"):
    split_dir = PROJECT_DIR / "data" / split
    assert split_dir.is_dir() and any(split_dir.glob("*.csv")), (
        f"Thiếu dữ liệu evaluation: {split_dir}"
    )

# Ba benchmark retrieval (~90 MB) không nằm trong git. Chúng cũng là exclusion set
# của bước dựng corpus, nên phải có trước cả khi build training data.
missing_retrieval = [
    name
    for name in ("arguana", "fiqa", "scidocs")
    if not (RETRIEVAL_DIR / name / "corpus.csv").is_file()
]
if missing_retrieval and (EVAL_RETRIEVAL or DATASET_CONFIG["build"]):
    if not AUTO_FETCH_DATA:
        raise FileNotFoundError(
            f"Thiếu benchmark retrieval {missing_retrieval}. Chạy: "
            f"python3 scripts/download_retrieval_benchmarks.py "
            f"(hoặc bật AUTO_FETCH_DATA / tắt EVAL_RETRIEVAL)"
        )
    run_script(["scripts/download_retrieval_benchmarks.py"], "tải benchmark retrieval")

if not TRAIN_DATA.is_file():
    if DATASET_CONFIG["build"] is None:
        raise FileNotFoundError(
            f"Không tìm thấy training data của DATASET={DATASET!r}: {TRAIN_DATA}. "
            f"File này lẽ ra nằm sẵn trong repo."
        )
    if not AUTO_FETCH_DATA:
        raise FileNotFoundError(
            f"Không tìm thấy training data của DATASET={DATASET!r}: {TRAIN_DATA}. "
            f"Dựng lại bằng: python3 {DATASET_CONFIG['build']}"
        )
    # Bước này tải một shard MS MARCO ~240 MB ở lần chạy đầu, sau đó dùng cache.
    run_script(DATASET_CONFIG["build"].split(), f"dựng corpus {DATASET}")
assert TRAIN_DATA.is_file(), f"Dựng corpus xong nhưng vẫn thiếu: {TRAIN_DATA}"

if not torch.cuda.is_available():
    raise RuntimeError(f"{TEACHER_MODEL} cần GPU; hãy bật GPU runtime rồi chạy lại.")
if hasattr(torch.cuda, "is_bf16_supported") and not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "Config của repo tải teacher bằng BF16 nhưng GPU hiện tại không hỗ trợ BF16. "
        "Hãy chọn A100, L4 hoặc GPU Ampere/newer."
    )

run_root_existed = RUN_ROOT.is_dir()
RUN_ROOT.mkdir(parents=True, exist_ok=True)
if run_root_existed:
    # Chạy tiếp vào một run cũ (ví dụ chạy bổ sung method còn thiếu). An toàn vì
    # cell 5 bỏ qua method đã có final test và dừng hẳn khi gặp metrics dở dang,
    # nên không có chuyện hai cấu hình khác nhau cùng ghi vào một thư mục.
    print(f"[note] RUN_ROOT đã tồn tại, sẽ chạy tiếp vào run cũ: {RUN_ROOT}")
print(f"PyTorch: {torch.__version__}; CUDA build: {torch.version.cuda}")
print(f"Visible GPUs before subprocess filtering: {torch.cuda.device_count()}")
largest_gib = 0.0
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    largest_gib = max(largest_gib, props.total_memory / 2**30)
    print(f"  cuda:{index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
if largest_gib < PAIR_CONFIG["min_vram_gib"]:
    print(
        f"[WARN] Cặp {PAIR} được ước lượng cần >= {PAIR_CONFIG['min_vram_gib']} GiB "
        f"trên một GPU cho các method online; GPU lớn nhất hiện có {largest_gib:.1f} GiB."
    )
train_rows = sum(1 for _ in TRAIN_DATA.open(encoding="utf-8")) - 1
print(f"Training data: {TRAIN_DATA} ({train_rows} dòng)")
print(f"Retrieval eval: {'bật' if EVAL_RETRIEVAL else 'tắt'}")
print(f"Output root: {RUN_ROOT}")


: 

In [ ]:
# 4. Tạo và hiển thị command của từng method để kiểm tra trước khi train.
import shlex

# Cache teacher nằm NGOÀI run và dùng chung cho mọi run: chạy lại teacher trên 100k
# câu là việc đắt nhất trong pipeline và nó không đổi giữa các run của cùng một cặp.
# Tên file do --cache_dir tự sinh từ những thứ quyết định cache có dùng lại được hay
# không (teacher, pooling, normalize, max_length và *nội dung* corpus), nên một thư
# mục chứa được mọi cache: run hoặc tìm đúng cache của mình, hoặc miss — không bao
# giờ nạp nhầm cache của cặp khác rồi bị từ chối.
# Trên Colab có mount Drive thì OUTPUT_BASE nằm trong Drive, nên cache sống qua cả
# các session sau.
CACHE_DIR = OUTPUT_BASE / "teacher_cache"


def build_command(name, settings):
    """Command của một dòng trong bảng. `name` là tên dòng/thư mục; `--method` là
    settings["method"] nếu dòng là biến thể, còn không thì chính là `name`."""
    method = settings.get("method", name)
    method_dir = RUN_ROOT / name
    command = [
        sys.executable,
        str(PROJECT_DIR / "main.py"),
        "--method", method,
        "--train_data", str(TRAIN_DATA),
        "--student_model", STUDENT_MODEL,
        # Teacher được truyền cho MỌI method, kể cả simcse. SimCSE-only không tải
        # teacher (distiller bỏ qua nó theo thiết kế — đây là control không distill),
        # nhưng config của run ghi lại teacher + pooling để bảng kết quả nói rõ nó là
        # control của cặp nào; đổi PAIR thì dòng simcse cũng đổi theo.
        "--teacher_model", TEACHER_MODEL,
        "--teacher_pooling", TEACHER_POOLING,
        "--batch_size", str(settings["batch_size"]),
        "--epochs", str(EPOCHS),
        "--save_every", str(EPOCHS),
        "--lr", str(settings["learning_rate"]),
        "--max_length", str(MAX_LENGTH),
        "--save_dir", str(method_dir),
        "--num_workers", str(NUM_WORKERS),
        "--pair_threshold_source", PAIR_THRESHOLD_SOURCE,
        "--eval_every", str(EVAL_EVERY),
        "--no_wandb",
    ]
    if EVAL_ON_TEST_EACH_EPOCH:
        command.append("--evaluate_test_each_epoch")
    if not EVAL_RETRIEVAL:
        command.append("--no_eval_retrieval")
    if method == "cdm":
        # Marker sub-word của tokenizer teacher, để DTW so token string đúng.
        command.extend(["--teacher_special_token", TEACHER_SPECIAL_TOKEN])
    if method == "emo" and EMO_TEACHER_SPECIAL_TOKEN is not None:
        # EMO đọc flag này như BOS token của teacher (chỉ có nghĩa với encoder teacher).
        command.extend(["--teacher_special_token", EMO_TEACHER_SPECIAL_TOKEN])
    if method in ("talas", "geoode", "rkd"):
        # All three read the same teacher pooling over the same corpus, so they
        # share one cache file instead of running the teacher three times -- and so
        # does every later run of the same pair. The file is keyed by the teacher, the pooling and the corpus
        # contents, so a cache built for another pair is a miss rather than a
        # refusal, and a corpus rebuilt in place is a miss rather than stale data.
        command.extend(["--cache_dir", str(CACHE_DIR)])
    # Flag riêng của biến thể (ví dụ pca_mse) đi sau cùng để thắng mọi default.
    command.extend(settings.get("args", []))
    return command


COMMANDS = {
    name: build_command(name, settings)
    for name, settings in METHOD_SETTINGS.items()
}
for name, command in COMMANDS.items():
    print(f"[{name.upper()}] {shlex.join(command)}\n")
print("Lưu ý: Stella dùng lịch mặc định 2 epoch stage 1 + 3 epoch stage 2.")
print("Lưu ý: simcse không tải teacher — nó là control không distill của bảng kết quả.")
print("Lưu ý: pca_mse là recipe sentence-transformers <= v5.4 chạy qua code path geoode (loss MSE, không InfoNCE, không gauge).")


: 

In [ ]:
# 5. Chạy tuần tự tất cả method và tee stdout/stderr vào train.log.
import json
import time

# Notebook chỉ hiện dòng progress mới nhất, tối đa mỗi N giây, và cắt bớt độ dài.
# Log đầy đủ vẫn được ghi vào <method>_train.log.
PROGRESS_EVERY_SEC = 30
PROGRESS_MAX_CHARS = 160


def has_final_test(metrics_path):
    """True only for the end-of-run record.

    With EVAL_ON_TEST_EACH_EPOCH and EVAL_EVERY > 0 every epoch also writes a "test"
    payload, so the end-of-run record is identified by what it lacks: it carries no
    "train" block.
    """
    if not metrics_path.is_file():
        return False
    with metrics_path.open(encoding="utf-8") as handle:
        return any(
            json.loads(line).get("test") and not json.loads(line).get("train")
            for line in handle
            if line.strip()
        )


env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
env["TOKENIZERS_PARALLELISM"] = "false"
env["WANDB_MODE"] = "disabled"
# tqdm ghi mỗi ~0.1s một dòng "\r..." vào pipe; notebook in hết sẽ rất lag.
# Giảm tần suất tqdm từ phía subprocess, và bên dưới chỉ in progress mới nhất.
env["TQDM_MININTERVAL"] = str(PROGRESS_EVERY_SEC)
run_status = []


def stream_output(stream, log_handle):
    """Tee subprocess output: full log ra file, notebook chỉ hiện progress mới nhất.

    tqdm tách các lần cập nhật bằng "\r" thay vì "\n", nên đọc theo từng đoạn
    và tự tách theo cả hai ký tự. Dòng progress (chứa "%|") được ghi đè tại
    chỗ và chỉ in tối đa mỗi PROGRESS_EVERY_SEC giây; dòng thường in ngay.
    """
    buffer = ""
    last_progress = 0.0
    progress_shown = False
    while True:
        chunk = stream.read(4096)
        if not chunk:
            break
        buffer += chunk
        parts = buffer.replace("\r\n", "\n").replace("\r", "\n").split("\n")
        buffer = parts.pop()
        for line in parts:
            log_handle.write(line + "\n")
            if "%|" in line:
                now = time.perf_counter()
                if now - last_progress >= PROGRESS_EVERY_SEC:
                    print("\r" + line[:PROGRESS_MAX_CHARS].ljust(PROGRESS_MAX_CHARS), end="", flush=True)
                    last_progress = now
                    progress_shown = True
            elif line.strip():
                if progress_shown:
                    print()
                    progress_shown = False
                print(line)
        log_handle.flush()
    if buffer:
        log_handle.write(buffer + "\n")
        if "%|" not in buffer:
            print(buffer)
    if progress_shown:
        print()

for position, (method, command) in enumerate(COMMANDS.items(), start=1):
    method_dir = RUN_ROOT / method
    metrics_path = method_dir / "metrics.jsonl"
    log_path = RUN_ROOT / f"{method}_train.log"
    if has_final_test(metrics_path):
        print(f"[SKIP] {method.upper()} đã có final test: {metrics_path}")
        run_status.append({"method": method, "status": "skipped_complete", "seconds": 0.0})
        continue
    if metrics_path.exists():
        raise RuntimeError(
            f"{method} có run dở dang tại {method_dir}. "
            "Dùng RUN_NAME mới để tránh nối metrics vào run cũ."
        )

    print("\n" + "#" * 88)
    print(f"METHOD {position}/{len(COMMANDS)}: {method.upper()}")
    print(f"Log: {log_path}")
    print("#" * 88)
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            cwd=PROJECT_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        assert process.stdout is not None
        stream_output(process.stdout, log_handle)
        return_code = process.wait()
    elapsed = time.perf_counter() - started
    status = "complete" if return_code == 0 and has_final_test(metrics_path) else "failed"
    run_status.append({"method": method, "status": status, "seconds": elapsed})
    print(f"[{status.upper()}] {method} in {elapsed / 60:.1f} minutes")
    if status == "failed" and STOP_ON_ERROR:
        raise RuntimeError(f"Method {method} failed; xem log tại {log_path}")

print("\nRun status:")
for item in run_status:
    print(f"  {item['method']:8s} {item['status']:18s} {item['seconds'] / 60:8.1f} min")


: 

In [ ]:
# 6. Tổng hợp validation theo epoch, final test và chi tiết từng benchmark.
import pandas as pd
from IPython.display import display


def benchmark_name(path, split):
    name = Path(path).stem
    suffix = f"_{split}"
    return name[:-len(suffix)] if name.endswith(suffix) else name


def detailed_rows(method, split, epoch, payload):
    rows = []
    primary_metrics = {
        "classification": "f1",
        "pair": "average_precision",
        "sts": "spearman",
        "retrieval": "ndcg_at_10",
    }
    for family, metric_name in primary_metrics.items():
        for path, raw_values in payload.get(family, {}).items():
            values = {"spearman": raw_values} if family == "sts" else dict(raw_values)
            rows.append({
                "method": method,
                "split": split,
                "epoch": epoch,
                "family": family,
                "benchmark": benchmark_name(path, split),
                "primary_metric": metric_name,
                "primary_score": float(values[metric_name]),
                **{key: float(value) for key, value in values.items() if isinstance(value, (int, float))},
            })
    return rows


# avg_retrieval chỉ xuất hiện ở record test (retrieval không chấm trên validation),
# nên nó được đọc như một cột có thể trống thay vì bắt buộc.
SUMMARY_KEYS = ("avg_iod", "avg_ood", "avg_retrieval", "avg_all")

epoch_summary_rows = []
final_summary_rows = []
detail_rows = []
for method in METHOD_SETTINGS:
    metrics_path = RUN_ROOT / method / "metrics.jsonl"
    if not metrics_path.is_file():
        print(f"[WARN] Missing metrics for {method}: {metrics_path}")
        continue
    with metrics_path.open(encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    for record in records:
        train = record.get("train")
        split = "validation" if record.get("validation") else "test"
        payload = record.get(split)
        if not payload:
            continue
        summary = payload["summary"]
        if train is not None:
            # Per-epoch record: on the split this run was configured to evaluate.
            epoch = int(train.get("epoch", 0))
            epoch_summary_rows.append({
                "method": method,
                "split": split,
                "stage": record.get("stage"),
                "epoch": epoch,
                "train_loss": train.get("loss"),
                **{
                    key: (None if summary.get(key) is None else float(summary[key]))
                    for key in SUMMARY_KEYS
                },
            })
            detail_rows.extend(detailed_rows(method, split, epoch, payload))
        else:
            # End-of-run record: no "train" block.
            final_summary_rows.append({
                "method": method,
                **{
                    key: (None if summary.get(key) is None else float(summary[key]))
                    for key in SUMMARY_KEYS
                },
            })
            detail_rows.extend(detailed_rows(method, "test", None, payload))

eval_by_epoch = pd.DataFrame(epoch_summary_rows)
final_test_results = pd.DataFrame(final_summary_rows)
benchmark_details = pd.DataFrame(detail_rows)
assert not final_test_results.empty, "Chưa có final test result nào để tổng hợp."
eval_by_epoch.to_csv(RUN_ROOT / "eval_by_epoch.csv", index=False)
final_test_results.to_csv(RUN_ROOT / "final_test_results.csv", index=False)
benchmark_details.to_csv(RUN_ROOT / "benchmark_details.csv", index=False)
pd.DataFrame(run_status).to_csv(RUN_ROOT / "run_status.csv", index=False)

epoch_split = eval_by_epoch["split"].iloc[0].upper() if not eval_by_epoch.empty else "-"
print(f"{epoch_split} BY EPOCH")
display(eval_by_epoch.style.format(precision=4))
print("FINAL TEST COMPARISON")
display(final_test_results.sort_values("avg_all", ascending=False).style.format(precision=4))
print(f"Saved summary files to: {RUN_ROOT}")


: 

In [ ]:
# 7. Vẽ comparison plots và lưu PNG cùng kết quả.
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
if not eval_by_epoch.empty:
    split_label = eval_by_epoch["split"].iloc[0]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for method, frame in eval_by_epoch.groupby("method", sort=False):
        frame = frame.sort_values("epoch")
        ax.plot(frame["epoch"], frame["avg_all"], marker="o", label=method.upper())
    ax.set(title=f"{split_label.capitalize()} average across all benchmarks",
           xlabel="epoch", ylabel="avg_all")
    ax.legend(ncol=3, frameon=False)
    fig.tight_layout()
    fig.savefig(RUN_ROOT / "eval_avg_all.png", dpi=180, bbox_inches="tight")
    plt.show()

ordered = final_test_results.sort_values("avg_all", ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(ordered["method"].str.upper(), ordered["avg_all"], color="#2a78d6")
ax.bar_label(ax.containers[0], fmt="%.4f", padding=4)
ax.set(title="Final test comparison", xlabel="average score across all benchmarks", ylabel="")
ax.set_xlim(0, min(1.0, float(ordered["avg_all"].max()) * 1.15))
fig.tight_layout()
fig.savefig(RUN_ROOT / "final_test_comparison.png", dpi=180, bbox_inches="tight")
plt.show()


: 